In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# get data from classified dataset
df = pd.read_csv("../data/cpdb_master_with_classification.csv")

# Create 'Feature' column
df['text_features'] = (
    df['policy_name'].fillna('') + " " + 
    df['policy_instrument'].fillna('') + " " + 
    df['policy_description'].fillna('')
)

# Filter out rows where classification is 'Neutral/Unclassified'
df = df[df['classification'] != 'Neutral/Unclassified']

In [ ]:
print(df.head())

In [ ]:
# Define X (input) and y (target)
from sklearn.model_selection import train_test_split


X = df['text_features']
y = df['classification']

# Split into 80% Training and 20% Testing
# stratify=y ensures both sets have the same ratio of classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create the Pipeline
# ngram_range=(1, 2) helps the model see "Carbon Tax" as one concept
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1, 2))),
    ('clf', LogisticRegression(class_weight='balanced'))
])

# Train
print("Training the model...")
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
print("\nModel Performance Report:")
print(classification_report(y_test, y_pred))

Training the model...

Model Performance Report:
                    precision    recall  f1-score   support

Likely Progressive       0.85      0.95      0.90        42
 Likely Regressive       0.89      0.70      0.78        23

          accuracy                           0.86        65
         macro avg       0.87      0.82      0.84        65
      weighted avg       0.86      0.86      0.86        65



In [9]:
# Prepare the text features for the old dataset

df_old = pd.read_csv("../data/cpdb_master_with_classification.csv")  # Load the original dataset

df_old['text_features'] = (
    df_old['policy_name'].fillna('') + " " + 
    df_old['policy_instrument'].fillna('') + " " + 
    df_old['policy_description'].fillna('')
)

# Use the trained model to predict
df_old['ml_prediction'] = model_pipeline.predict(df_old['text_features'])

# Create a "Conflict" column to see where the ML disagrees with original labels
df_old['is_mismatch'] = df_old['classification'] != df_old['ml_prediction']

# Filter to see the mismatches
mismatches = df_old[df_old['is_mismatch'] == True]

print(f"Total policies analyzed: {len(df_old)}")
print(f"Number of mismatches found: {len(mismatches)}")
save_path = "../data/mismatches.csv"
mismatches.to_csv(save_path, index=False)
print(f"Mismatches saved to {save_path}")

# Save entire new dataset with policy name, description, original classification, ML prediction, and mismatch flag
full_save_path = "../data/full_predictions.csv"
# Take out colums we dont need
df_old = df_old[[
    'policy_name', 
    'policy_instrument', 
    'policy_description', 
    'classification', 
    'ml_prediction', 
    'is_mismatch'
]]
df_old.to_csv(full_save_path, index=False)
print(f"Full predictions saved to {full_save_path}")

Total policies analyzed: 733
Number of mismatches found: 421
Mismatches saved to ../data/mismatches.csv
Full predictions saved to ../data/full_predictions.csv


In [10]:
print(df_old.head())

                                         policy_name  \
0  Intended Nationally Determined Contribution - ...   
1  Second Economic Development and Poverty Reduct...   
2  Law No. 16 of 22 May 2012, determining the Org...   
3  Green Growth and Climate Resilience National S...   
4             Kigali Amendment on HFCs Rwanda (2017)   

                                   policy_instrument  \
0                                     Policy support   
1                 Policy support, Strategic planning   
2             Policy support, Institutional creation   
3  Climate strategy, Political &amp; non-binding ...   
4                       Target, GHG reduction target   

                                  policy_description        classification  \
0  As it is a small developing nation, the INDC o...  Neutral/Unclassified   
1                                          Executive  Neutral/Unclassified   
2                                        Legislative    Likely Progressive   
3  Executive\n